# PTP through networks without PTP support: servo, filtering or transparent clocks?

A PTP slave computes its offset from every Sync with the latest mean path delay. Across switches
that do not support PTP, each packet waits a different time in the queues (packet delay variation,
PDV), and that variation goes straight into the offset. Three things can help: a slower servo, an
estimator that uses each exchange's delay to pick the least-delayed packets, or transparent clocks
that measure the residence time and put it in the correction field.

This notebook compares them on simulated PTP exchanges with known truth (ntpstats 3.1+): the
`ptp-lan` preset (1 Sync and 1 Delay_Req per second, hardware time stamps, switches without PTP
support, more queueing towards the master) and `ptp-tc` (the same network with transparent clocks
that correct 95 % of the queueing). The client models are written from the published algorithms
(linuxptp, Meta's SPTP); they show the mechanisms, not the behaviour of a particular release.

In [ ]:
from dataclasses import replace
from pathlib import Path

import numpy as np

from ntpstats import api as nt
from ntpstats.bench import summarize
from ntpstats.estimators import ptp4l
from ntpstats.simulate import PRESETS

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())
fmt = nt.format_seconds

## What the slave sees

Two hours of `ptp-lan`. The raw offset is what a slave computes from each Sync; `true_offset` is the
ground truth. Its error is the queueing: one-sided (packets are only ever delayed), and larger
towards the master, which also biases the mean.

In [ ]:
sc = replace(PRESETS["ptp-lan"], duration=7200, seed=1)
meas, truth = nt.simulate_ntp(sc)
err = meas.offset - meas.extra["true_offset"]
print(f"{len(meas)} Syncs, Delay_Req every {sc.delay_interval or sc.poll:g} s")
print(f"raw offset error: rms {fmt(np.sqrt(np.mean(err**2)))}, mean {fmt(np.mean(err))}, "
      f"p99 |e| {fmt(np.percentile(np.abs(err), 99))}")

## Every client on the same exchanges

The bench scores each estimator against the truth after a 30-minute warm-up. `ptp4l` uses the
moving median of the last 10 path delays and linuxptp's PI servo with its default gains; `sptp`
uses complete exchanges and discards those whose path delay is an outlier; `mindelay` and `hull`
are the NTP clock filter and a Huygens-style estimator, applied to the same exchanges.

In [ ]:
scen = [(n, replace(PRESETS[n], duration=7200)) for n in ("ptp-lan", "ptp-tc")]
rows = nt.run_bench(scen, ["raw", "ptp4l", "sptp", "kalman-dw", "mindelay", "hull"], seeds=[1, 2])
table = summarize(rows)
print(f"{'estimator':<11} {'ptp-lan rms':>12} {'ptp-tc rms':>12}")
for est in ("raw", "ptp4l", "sptp", "kalman-dw", "mindelay", "hull"):
    r = {t["scenario"]: t for t in table if t["estimator"] == est}
    print(f"{est:<11} {fmt(r['ptp-lan']['rms']):>12} {fmt(r['ptp-tc']['rms']):>12}")

With linuxptp's default gains at one Sync per second (kp = 0.7, ki = 0.3) the servo's bandwidth is
wide: it follows most of the PDV, so `ptp4l` is barely better than the raw offsets. Transparent
clocks remove the cause and improve every client about twentyfold. Estimators that look at each
exchange's delay get much closer from the very same packets, even without transparent clocks.

## How much transparent-clock coverage is enough?

The fraction of queueing that the correction field removes, from none to full on-path support.

In [ ]:
print(f"{'corrected':>9} {'ptp4l rms':>10} {'mindelay rms':>13}")
for frac in (0.0, 0.5, 0.9, 0.99):
    s = replace(PRESETS["ptp-lan"], duration=7200, transparent=frac)
    r = summarize(nt.run_bench([("x", s)], ["ptp4l", "mindelay"], seeds=[1, 2]))
    by = {t["estimator"]: t["rms"] for t in r}
    print(f"{frac:>9.0%} {fmt(by['ptp4l']):>10} {fmt(by['mindelay']):>13}")

The servo's error falls roughly in proportion to the uncorrected part of the queueing: a partial
deployment helps, but only full coverage brings it to the timestamp noise.

## A narrower servo

A slower loop averages more Syncs. The price is a slower response to the oscillator's own wander
and a longer lock time; here the oscillator is good (white FM 1e-10 at 1 s), so the trade is
favourable.

In [ ]:
print(f"{'kp':>6} {'ki':>8} {'rms':>9}")
for kp, ki in ((None, None), (0.1, 0.005), (0.03, 4e-4)):
    errs = []
    for seed in (1, 2):
        m, tr = nt.simulate_ntp(replace(PRESETS["ptp-lan"], duration=7200, seed=seed))
        opts = {} if kp is None else {"kp": kp, "ki": ki}
        est = ptp4l(m, servo_options=opts)
        late = est.t > est.t[0] + 1800
        errs.append(est.offset[late] - np.interp(est.t[late], tr.t, tr.offset))
    e = np.concatenate(errs)
    print(f"{'0.7' if kp is None else kp:>6} {'0.3' if ki is None else ki:>8} {fmt(np.sqrt(np.mean(e**2))):>9}")

## The same correction for NTP: RFC 10030

NTP over PTP (RFC 10030, chrony 4.9) carries NTP in PTP event messages so that transparent clocks
correct it too. `examples/data/ntp-over-ptp.pcap` is a synthetic client-side capture through two
transparent clocks; ntpstats applies the corrections as the RFC specifies and keeps the
uncorrected values.

In [ ]:
(cap,) = nt.load(str(DATA / "ntp-over-ptp.pcap"))
for col, label in (("offset_uncorrected", "uncorrected"), (None, "corrected")):
    x = cap.extra[col] if col else cap.offset
    d = np.diff(x)
    print(f"{label:>11}: offset std {fmt(np.std(x))}, sample-to-sample std {fmt(np.std(d))}")
print("corrected exchanges:", cap.meta["corrected"], "of", len(cap))

## Take-aways

- Through switches without PTP support, a slave with default servo gains passes the PDV through:
  microseconds here.
- Transparent clocks remove the cause; partial coverage helps in proportion.
- Filtering by delay (or a narrower servo) recovers much of the accuracy from the same packets.
  Asymmetric *floors* (unequal propagation delays) remain invisible to every two-way method.

Reproduce or extend: change the preset's paths, rates or oscillator, add your own estimator
(see the custom-estimator notebook), or replay a real capture's delays (`trace:FILE`).